## Phase 3 - Compact Feature and Target Upgrade

This notebook materializes compact but smarter modeling tables and runs a focused sensitivity study.

Goals in this phase:
1. Add a small set of higher-value features without exploding feature count.
2. Compare explicit regression target definitions.
3. Test heart-rate fill strategy sensitivity.
4. Save preferred task-specific processed tables and ablation artifacts.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from pamap2_telemetry.ablation import run_compact_ablation_study

METRICS_DIR = REPO_ROOT / "artifacts" / "metrics"
REGRESSION_PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table_regression.parquet"
CLASSIFICATION_PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table_classification.parquet"

print(f"Repo root: {REPO_ROOT}")
print(f"Regression-ready table exists before run: {REGRESSION_PROCESSED_PATH.exists()}")
print(f"Classification-ready table exists before run: {CLASSIFICATION_PROCESSED_PATH.exists()}")


In [ ]:
results = run_compact_ablation_study(repo_root=REPO_ROOT)

feature_ablation_df = results["feature_ablation"].copy()
target_comparison_df = results["target_comparison"].copy()
fill_sensitivity_df = results["fill_sensitivity"].copy()
preferred_setup_df = results["preferred_setup"].copy()

print("Compact feature/target/fill study complete.")
display(preferred_setup_df)


### Feature Ablation

This section compares the compact baseline feature set with the upgraded feature set while holding target and fill policy constant.


In [ ]:
display(feature_ablation_df.sort_values("best_mean_mae").reset_index(drop=True))

baseline_row = feature_ablation_df[feature_ablation_df["feature_set"] == "baseline"].iloc[0]
upgraded_row = feature_ablation_df[feature_ablation_df["feature_set"] == "upgraded"].iloc[0]
delta_mae = baseline_row["best_mean_mae"] - upgraded_row["best_mean_mae"]

print(f"MAE improvement from upgraded feature set: {delta_mae:.3f} bpm")
print(f"Baseline feature count: {int(baseline_row['feature_count'])}")
print(f"Upgraded feature count: {int(upgraded_row['feature_count'])}")


### Target And Fill Sensitivity

This section isolates target-definition changes and fill-policy changes using the upgraded feature set.


In [ ]:
print("Target comparison:")
display(target_comparison_df.sort_values("best_mean_mae").reset_index(drop=True))

print("Fill strategy sensitivity:")
display(fill_sensitivity_df.sort_values("best_mean_mae").reset_index(drop=True))

print("Preferred setup:")
display(preferred_setup_df)


### Final Processed Tables And Feature Inventory

These checks confirm the saved task-specific modeling tables and list the final preferred feature inventory.


In [ ]:
feature_summary_path = METRICS_DIR / "grouped_cv_final_feature_summary.csv"
preferred_setup_path = METRICS_DIR / "grouped_cv_preferred_setup_summary.csv"
row_summary_path = METRICS_DIR / "grouped_cv_task_table_row_summary.csv"

regression_model_df = pd.read_parquet(REGRESSION_PROCESSED_PATH)
classification_model_df = pd.read_parquet(CLASSIFICATION_PROCESSED_PATH)
feature_summary_df = pd.read_csv(feature_summary_path)
preferred_setup_saved_df = pd.read_csv(preferred_setup_path)
row_summary_df = pd.read_csv(row_summary_path)

print(f"Saved regression-ready table: {REGRESSION_PROCESSED_PATH}")
print(f"Saved classification-ready table: {CLASSIFICATION_PROCESSED_PATH}")
print(f"Regression rows: {len(regression_model_df):,}")
print(f"Classification rows: {len(classification_model_df):,}")
print(
    "Classification row gain vs regression:",
    int(row_summary_df.iloc[0]["classification_row_gain_vs_regression"]),
)
print(f"Column count (shared schema): {len(regression_model_df.columns)}")
print("Target columns:", [c for c in regression_model_df.columns if c.startswith("hr_target_")])
print("Fill strategy values:", sorted(regression_model_df["heart_rate_fill_strategy"].unique().tolist()))

print("\nSaved preferred setup:")
display(preferred_setup_saved_df)

print("\nTask-table row summary:")
display(row_summary_df)

print("\nFinal feature inventory sample:")
display(feature_summary_df.head(20))


In [ ]:
artifact_paths = [
    METRICS_DIR / "grouped_cv_feature_ablation_summary.csv",
    METRICS_DIR / "grouped_cv_target_comparison_summary.csv",
    METRICS_DIR / "grouped_cv_fill_sensitivity_summary.csv",
    METRICS_DIR / "grouped_cv_preferred_setup_summary.csv",
    METRICS_DIR / "grouped_cv_final_feature_summary.csv",
    METRICS_DIR / "grouped_cv_task_table_row_summary.csv",
]

print("New compact-ablation metric artifacts:")
for artifact_path in artifact_paths:
    print("-", artifact_path.relative_to(REPO_ROOT), "| exists=", artifact_path.exists())


## Phase 3 Completion Summary

This notebook now completes the compact upgrade and sensitivity goals:
1. targeted feature upgrades were added with a controlled feature count,
2. explicit target variants were compared,
3. heart-rate fill strategies were stress-tested,
4. the preferred setup was selected and saved to task-specific tables:
   - `data/processed/pamap2_model_table_regression.parquet`
   - `data/processed/pamap2_model_table_classification.parquet`
5. ablation and feature-summary artifacts were saved to `artifacts/metrics/` and `artifacts/figures/`.
